In [4]:
# Load BESSTIE dataset and view overall stats
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

print(dataset)

training_dataframe = dataset['train'].to_pandas()
print(training_dataframe['Sentiment'].value_counts())
print(training_dataframe['Sarcasm'].value_counts())
print(training_dataframe['variety'].value_counts())




DatasetDict({
    train: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 3747
    })
    validation: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 313
    })
    test: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 2183
    })
})
Sentiment
0.0    1907
1.0    1840
Name: count, dtype: int64
Sarcasm
0.0    3223
1.0     524
Name: count, dtype: int64
variety
en-IN    1399
en-UK    1203
en-AU    1145
Name: count, dtype: int64


In [5]:
vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(training_dataframe['text'])
X_test = vectorizer.transform(dataset['test'].to_pandas()['text'])

# Sentiment
lr_sentiment = LogisticRegression(max_iter=1000)
lr_sentiment.fit(X_train, training_dataframe['Sentiment'])
sentiment_predictions = lr_sentiment.predict(X_test)

test_dataframe = dataset['test'].to_pandas()
print("LR Sentiment metrics", classification_report(test_dataframe['Sentiment'], sentiment_predictions))

# Sarcasm
lr_sarcasm = LogisticRegression(max_iter=1000)
lr_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
sarcasm_predictions = lr_sarcasm.predict(X_test)
print("LR Sarcasm metrics", classification_report(test_dataframe['Sarcasm'], sarcasm_predictions))

LR Sentiment metrics               precision    recall  f1-score   support

         0.0       0.81      0.88      0.84      1117
         1.0       0.86      0.79      0.82      1066

    accuracy                           0.83      2183
   macro avg       0.83      0.83      0.83      2183
weighted avg       0.83      0.83      0.83      2183

LR Sarcasm metrics               precision    recall  f1-score   support

         0.0       0.86      1.00      0.92      1878
         1.0       0.00      0.00      0.00       305

    accuracy                           0.86      2183
   macro avg       0.43      0.50      0.46      2183
weighted avg       0.74      0.86      0.80      2183



These results show that accuracy is the wrong metric. For sarcasm the dataset is highly imbalanced. The 0.86 accuracy is misleading. The precision and recall for sarcasm is 0.00. The macro score of 0.46 is more useful.

In [6]:
# Balanced class weight
lr_sarcasm = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
sarcasm_predictions = lr_sarcasm.predict(X_test)
print(classification_report(test_dataframe['Sarcasm'], sarcasm_predictions))

              precision    recall  f1-score   support

         0.0       0.93      0.77      0.85      1878
         1.0       0.32      0.66      0.43       305

    accuracy                           0.76      2183
   macro avg       0.63      0.71      0.64      2183
weighted avg       0.85      0.76      0.79      2183



The model is now detecting sarcasm. The accuracy is lower but the macro and weighted averages are higher.

In [7]:
from sklearn.svm import LinearSVC

# Sentiment
svm_sentiment = LinearSVC(max_iter=10000)
svm_sentiment.fit(X_train, training_dataframe['Sentiment'])
print("SVM Sentiment metrics:")
print(classification_report(test_dataframe['Sentiment'], svm_sentiment.predict(X_test)))

# Sarcasm
svm_sarcasm = LinearSVC(max_iter=10000, class_weight='balanced')
svm_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
print("SVM Sarcasm metrics:")
print(classification_report(test_dataframe['Sarcasm'], svm_sarcasm.predict(X_test)))

SVM Sentiment metrics:
              precision    recall  f1-score   support

         0.0       0.83      0.85      0.84      1117
         1.0       0.84      0.82      0.83      1066

    accuracy                           0.84      2183
   macro avg       0.84      0.84      0.84      2183
weighted avg       0.84      0.84      0.84      2183

SVM Sarcasm metrics:
              precision    recall  f1-score   support

         0.0       0.89      0.88      0.89      1878
         1.0       0.31      0.33      0.32       305

    accuracy                           0.81      2183
   macro avg       0.60      0.61      0.60      2183
weighted avg       0.81      0.81      0.81      2183



In [10]:
!pip install --upgrade transformers

RoBERTa sentiment analysis

In [23]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np


model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = dataset['train'].map(tokenize, batched=True)
tokenized_test = dataset['test'].map(tokenize, batched=True)
tokenized_val = dataset['validation'].map(tokenize, batched=True)

tokenized_train = tokenized_train.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])
tokenized_test = tokenized_test.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])
tokenized_val = tokenized_val.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])

tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_val.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="single_label_classification")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    from sklearn.metrics import classification_report
    print(classification_report(labels, preds))
    from sklearn.metrics import f1_score
    return {'macro_f1': f1_score(labels, preds, average='macro')}



trainer.train()

dtype after set_format (train): torch.int64
dtype after set_format (test): torch.int64
dtype after set_format (val): torch.int64


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.187547,0.932883
2,No log,0.224290,0.916932
3,0.280163,0.227690,0.929677


              precision    recall  f1-score   support

           0       0.94      0.93      0.93       160
           1       0.93      0.93      0.93       153

    accuracy                           0.93       313
   macro avg       0.93      0.93      0.93       313
weighted avg       0.93      0.93      0.93       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

              precision    recall  f1-score   support

           0       0.93      0.93      0.93       160
           1       0.93      0.93      0.93       153

    accuracy                           0.93       313
   macro avg       0.93      0.93      0.93       313
weighted avg       0.93      0.93      0.93       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=705, training_loss=0.233725505855912, metrics={'train_runtime': 458.1384, 'train_samples_per_second': 24.536, 'train_steps_per_second': 1.539, 'total_flos': 739407843325440.0, 'train_loss': 0.233725505855912, 'epoch': 3.0})

Test set evaluation

In [29]:
# 3 epochs is a common default for fine-tuning. We could try a higher number of epochs as a further investigation.
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

predictions = trainer.predict(tokenized_test)

preds = np.argmax(predictions.predictions, axis=-1)

# The predict method calls compute_metrics which prints the classification report
print(preds)

              precision    recall  f1-score   support

           0       0.90      0.86      0.88      1117
           1       0.86      0.91      0.88      1066

    accuracy                           0.88      2183
   macro avg       0.88      0.88      0.88      2183
weighted avg       0.88      0.88      0.88      2183

[1 1 0 ... 0 1 0]


Common methods for RoBERTa analysis

In [35]:
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

# Common methods
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    print(classification_report(labels, preds))
    return {'macro_f1': f1_score(labels, preds, average='macro')}

def prepare_dataset(split, task):
    tokenized = split.map(tokenize, batched=True)
    tokenized = tokenized.map(
        lambda examples: {'labels': [int(s) for s in examples[task]]},
        batched=True, remove_columns=[task]
    )
    tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return tokenized

# 3 epochs is a common default for fine-tuning. We could try a higher number of epochs as a further investigation.
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
)

RoBERTA Sarcasm analysis

In [33]:
# Sentiment
train_sentiment = prepare_dataset(dataset['train'], 'Sentiment')
test_sentiment = prepare_dataset(dataset['test'], 'Sentiment')
val_sentiment = prepare_dataset(dataset['validation'], 'Sentiment')

sentiment_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="single_label_classification")

trainer = Trainer(
    model=sentiment_model,
    args=training_args,
    train_dataset=train_sentiment,
    eval_dataset=val_sentiment,
    compute_metrics=compute_metrics,
)
trainer.train()
predictions = trainer.predict(test_sentiment)



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.213952,0.932632
2,No log,0.311185,0.894500
3,0.287810,0.236472,0.942421


              precision    recall  f1-score   support

           0       0.90      0.97      0.94       160
           1       0.97      0.89      0.93       153

    accuracy                           0.93       313
   macro avg       0.94      0.93      0.93       313
weighted avg       0.94      0.93      0.93       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.94      0.85      0.89       160
           1       0.86      0.94      0.90       153

    accuracy                           0.89       313
   macro avg       0.90      0.90      0.89       313
weighted avg       0.90      0.89      0.89       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.93      0.96      0.94       160
           1       0.95      0.93      0.94       153

    accuracy                           0.94       313
   macro avg       0.94      0.94      0.94       313
weighted avg       0.94      0.94      0.94       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

              precision    recall  f1-score   support

           0       0.90      0.90      0.90      1117
           1       0.89      0.90      0.90      1066

    accuracy                           0.90      2183
   macro avg       0.90      0.90      0.90      2183
weighted avg       0.90      0.90      0.90      2183



In [36]:
# Sarcasm
train_sarcasm = prepare_dataset(dataset['train'], 'Sarcasm')
test_sarcasm = prepare_dataset(dataset['test'], 'Sarcasm')
val_sarcasm = prepare_dataset(dataset['validation'], 'Sarcasm')

sarcasm_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="single_label_classification")

trainer_sarcasm = Trainer(
    model=sarcasm_model,
    args=training_args,
    train_dataset=train_sarcasm,
    eval_dataset=val_sarcasm,
    compute_metrics=compute_metrics,
)
trainer_sarcasm.train()
predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.413045,0.462199
2,No log,0.483179,0.462199
3,0.386420,0.399779,0.461274


              precision    recall  f1-score   support

           0       0.86      1.00      0.92       269
           1       0.00      0.00      0.00        44

    accuracy                           0.86       313
   macro avg       0.43      0.50      0.46       313
weighted avg       0.74      0.86      0.79       313



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

              precision    recall  f1-score   support

           0       0.86      1.00      0.92       269
           1       0.00      0.00      0.00        44

    accuracy                           0.86       313
   macro avg       0.43      0.50      0.46       313
weighted avg       0.74      0.86      0.79       313



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

              precision    recall  f1-score   support

           0       0.86      1.00      0.93      1878
           1       1.00      0.00      0.01       305

    accuracy                           0.86      2183
   macro avg       0.93      0.50      0.47      2183
weighted avg       0.88      0.86      0.80      2183



The results show a problem with the sarcasm model. Almost everything is being predicted as not sarcasm.

So the model needs adjusting to account for the class imbalance.